In [1]:
from itertools import permutations
from z3 import *

web_portal, add_services, site_stats, basic, advanced, site_search, images, text, html, dynamic, \
web_server, logging, db, file, protocol, http, ftp, https, cont, static, active, php, jsp, cgi, \
performance, ms, sec = Bools(
    "web_portal add_services site_stats basic advanced site_search images text html dynamic "
    "web_server logging db file protocol http ftp https cont static active php jsp cgi "
    "performance ms sec"
)

features = [
    web_portal, add_services, site_stats, basic, advanced, site_search, images, text, html, dynamic,
    web_server, logging, db, file, protocol, http, ftp, https, cont, static, active, php, jsp, cgi,
    performance, ms, sec
]

def fazerSolver():

    s = Solver()

    # -------------------------------------------------------------------------
    # 2. Restrições do Feature Model
    # -------------------------------------------------------------------------

    # web_portal é a raiz (root feature)[cite: 1]
    s.add(web_portal)

    # --- Filhos diretos de web_portal ---

    # add_services é opcional em web_portal[cite: 1, 2]
    s.add(Implies(add_services, web_portal))


    # web_server é obrigatório em web_portal[cite: 1, 2]
    s.add(web_server == web_portal)


    # performance é opcional em web_portal[cite: 1, 2]
    s.add(Implies(performance, web_portal))

    # --- Subárvore: add_services ---
    # site_stats e site_search são opcionais em add_services[cite: 1, 2]
    s.add(Implies(site_stats, add_services))
    s.add(Implies(site_search, add_services))

    # basic é obrigatório e advanced é opcional em site_stats[cite: 1, 2]
    s.add(basic == site_stats)
    s.add(Implies(advanced, site_stats))

    # images e text são opcionais em site_search[cite: 1, 2]
    s.add(Implies(images, site_search))
    s.add(Implies(text, site_search))

    # html é obrigatório e dynamic é opcional em text[cite: 1, 2]
    s.add(html == text)
    s.add(Implies(dynamic, text))

    # --- Subárvore: web_server ---
    # logging e protocol são opcionais em web_server[cite: 1, 2]
    s.add(Implies(logging, web_server))
    s.add(Implies(protocol, web_server))
    # cont é obrigatório em web_server[cite: 1, 2]
    s.add(cont == web_server)

    # db e file são sub-features XOR (Alternative) de logging[cite: 1, 2, 3]
    s.add(Or(db, file) == logging)
    s.add(Not(And(db, file)))

    # http, ftp e https são sub-features OR de protocol[cite: 1, 2, 3]
    s.add(Or(http, ftp, https) == protocol)

    # static é obrigatório e active é opcional em cont[cite: 1, 2]
    s.add(static == cont)
    s.add(Implies(active, cont))

    # php, jsp e cgi são sub-features OR de active[cite: 1, 2, 3]
    s.add(Or(php, jsp, cgi) == active)

    # --- Subárvore: performance ---
    # ms e sec são sub-features XOR (Alternative) de performance[cite: 1, 2, 3]
    s.add(Or(ms, sec) == performance)
    s.add(Not(And(ms, sec)))

    # -------------------------------------------------------------------------
    # 3. Cross-Tree Constraints (Setas tracejadas)
    # -------------------------------------------------------------------------
    # file requires ftp[cite: 1, 2, 3]
    s.add(Implies(file, ftp))

    # https excludes ms
    s.add(Not(And(https, ms)))

    # dynamic requires active[cite: 1, 2, 3]
    s.add(Implies(dynamic, active))
    return s

s = fazerSolver()

if s.check() == sat:
  print("Feature model is non void.")
else:
  print("Feature model is void.")

Feature model is non void.


In [9]:
s = fazerSolver()

s.push()
perm = permutations(features, 2)
resN = 0
res = None
lres = []
for (f1,f2) in perm:
    s.push()
    s.add(Implies(f1,f2))


    l = []
    c = 0
    for f in features:
        s.push()
        s.add(f)
        if s.check() == unsat:
            l.append(f)
            c += 1
            #print("Feature " + f.decl().name() + "is dead!")
        s.pop()
    if resN <= c:
        resN = c
        res = (f1,f2)
        lres = l
    s.pop()


print("feats: " + str(res) + " nr: " + str(resN) + " " + str(lres))

s.pop()
    

feats: (static, ms) nr: 2 [https, sec]


In [10]:
from z3 import *
from itertools import permutations

# Certifica-te de que 'features' está definido e 'fazerSolver()' retorna o solver configurado
s = fazerSolver()

s.push()
perm = permutations(features, 2)
resN = 0
res = None
lres = []

for (f1, f2) in perm:
    s.push()
    s.add(Implies(f1, f2))

    l = []
    c = 0
    for f in features:
        s.push()
        s.add(f) # Tenta forçar a feature a ser verdadeira
        if s.check() == unsat:
            l.append(f)
            c += 1
        s.pop()
        
    # Guarda o par que maximiza o número de conflitos
    if resN < c:
        resN = c
        res = (f1, f2)
        lres = l
        
    s.pop()

# Imprime os resultados com os nomes legíveis das features
f1_name, f2_name = res if res else (None, None)
dead_names = [f.decl().name() for f in lres]

print(f"feats: ({f1_name}, {f2_name}) nr: {resN} {dead_names}")

s.pop()

feats: (web_portal, ms) nr: 2 ['https', 'sec']
